In [69]:
import pandas as pd

In [70]:
df = pd.read_csv("data/raw_kolkata.csv", parse_dates=['date'])
df

,date,temperature,humidity,pressure
0,2016-01-01 18:30:00+00:00,16.111000,90.241035,1018.47614
1,2016-01-01 19:30:00+00:00,15.461000,91.371770,1017.87415
2,2016-01-01 20:30:00+00:00,14.961000,92.234870,1017.47210
3,2016-01-01 21:30:00+00:00,14.511001,92.811740,1017.37030
4,2016-01-01 22:30:00+00:00,14.461000,92.507000,1016.67084
...,...,...,...,...
70099,2023-12-31 13:30:00+00:00,17.900000,83.689150,1014.38940
70100,2023-12-31 14:30:00+00:00,17.850000,82.082560,1014.78890
70101,2023-12-31 15:30:00+00:00,17.550000,81.780350,1014.98740
70102,2023-12-31 16:30:00+00:00,16.350000,87.107796,1014.98160


In [71]:
df['label'] = 'normal'
df

,date,temperature,humidity,pressure,label
0,2016-01-01 18:30:00+00:00,16.111000,90.241035,1018.47614,normal
1,2016-01-01 19:30:00+00:00,15.461000,91.371770,1017.87415,normal
2,2016-01-01 20:30:00+00:00,14.961000,92.234870,1017.47210,normal
3,2016-01-01 21:30:00+00:00,14.511001,92.811740,1017.37030,normal
4,2016-01-01 22:30:00+00:00,14.461000,92.507000,1016.67084,normal
...,...,...,...,...,...
70099,2023-12-31 13:30:00+00:00,17.900000,83.689150,1014.38940,normal
70100,2023-12-31 14:30:00+00:00,17.850000,82.082560,1014.78890,normal
70101,2023-12-31 15:30:00+00:00,17.550000,81.780350,1014.98740,normal
70102,2023-12-31 16:30:00+00:00,16.350000,87.107796,1014.98160,normal


In [72]:
df.loc[1, 'temperature']

np.float64(15.461)

In [73]:
from weather_anamoly.utils import AnomalyInjection

In [74]:
anomaly_injector = AnomalyInjection()
df_anomaly = anomaly_injector.inject_anomalies(df=df.copy(), spike_rate=20, frozen_rate=50, comm_rate=150, drift_rate=6000)

In [75]:
df_anomaly.label.unique()

<StringArray>
['normal', 'spike', 'comm_error', 'frozen', 'fault']
Length: 5, dtype: str

In [76]:
print(df_anomaly['label'].eq('normal').sum())
print(df_anomaly['label'].eq('frozen').sum())
print(df_anomaly['label'].eq('comm_error').sum())
print(df_anomaly['label'].eq('spike').sum())
print(df_anomaly['label'].eq('fault').sum())

47031
8049
1159
2451
11414


In [77]:
df_anomaly.to_csv("data/anomaly_labelled_data.csv")

In [78]:
df = pd.read_csv("data/anomaly_labelled_data.csv", parse_dates=['date'], index_col=0).set_index('date')
df

,temperature,humidity,pressure,label
date,,,,
2016-01-01 18:30:00+00:00,16.111000,90.241035,1018.47614,normal
2016-01-01 19:30:00+00:00,15.461000,91.371770,1017.87415,normal
2016-01-01 20:30:00+00:00,14.961000,92.234870,1017.47210,normal
2016-01-01 21:30:00+00:00,14.511001,92.811740,1017.37030,normal
2016-01-01 22:30:00+00:00,14.461000,92.507000,1016.67084,normal
...,...,...,...,...
2023-12-31 13:30:00+00:00,17.900000,83.689150,1014.38940,normal
2023-12-31 14:30:00+00:00,17.850000,82.082560,1014.78890,normal
2023-12-31 15:30:00+00:00,17.550000,81.780350,1014.98740,normal


In [79]:
import numpy as np

In [80]:
df['hour'] = df.index.hour
df['month'] = df.index.month

df['cos_hour'] = np.cos(2*np.pi*df['hour']/24)
df['sin_hour'] = np.sin(2*np.pi*df['hour']/24)
df['cos_month'] = np.cos(2*np.pi*df['month']/12)
df['sin_month'] = np.sin(2*np.pi*df['month']/12)
df

,temperature,humidity,pressure,label,hour,month,cos_hour,sin_hour,cos_month,sin_month
date,,,,,,,,,,
2016-01-01 18:30:00+00:00,16.111000,90.241035,1018.47614,normal,18,1,-1.836970e-16,-1.000000,0.866025,5.000000e-01
2016-01-01 19:30:00+00:00,15.461000,91.371770,1017.87415,normal,19,1,2.588190e-01,-0.965926,0.866025,5.000000e-01
2016-01-01 20:30:00+00:00,14.961000,92.234870,1017.47210,normal,20,1,5.000000e-01,-0.866025,0.866025,5.000000e-01
2016-01-01 21:30:00+00:00,14.511001,92.811740,1017.37030,normal,21,1,7.071068e-01,-0.707107,0.866025,5.000000e-01
2016-01-01 22:30:00+00:00,14.461000,92.507000,1016.67084,normal,22,1,8.660254e-01,-0.500000,0.866025,5.000000e-01
...,...,...,...,...,...,...,...,...,...,...
2023-12-31 13:30:00+00:00,17.900000,83.689150,1014.38940,normal,13,12,-9.659258e-01,-0.258819,1.000000,-2.449294e-16
2023-12-31 14:30:00+00:00,17.850000,82.082560,1014.78890,normal,14,12,-8.660254e-01,-0.500000,1.000000,-2.449294e-16
2023-12-31 15:30:00+00:00,17.550000,81.780350,1014.98740,normal,15,12,-7.071068e-01,-0.707107,1.000000,-2.449294e-16


In [81]:
df['temp_gradient'] = df[['temperature']].diff()
df['pressure_gradient'] = df[['pressure']].diff()
df['humidity_gradient'] = df[['humidity']].diff()
df

,temperature,humidity,pressure,label,hour,month,cos_hour,sin_hour,cos_month,sin_month,temp_gradient,pressure_gradient,humidity_gradient
date,,,,,,,,,,,,,
2016-01-01 18:30:00+00:00,16.111000,90.241035,1018.47614,normal,18,1,-1.836970e-16,-1.000000,0.866025,5.000000e-01,NaN,NaN,NaN
2016-01-01 19:30:00+00:00,15.461000,91.371770,1017.87415,normal,19,1,2.588190e-01,-0.965926,0.866025,5.000000e-01,-0.650000,-0.60199,1.130735
2016-01-01 20:30:00+00:00,14.961000,92.234870,1017.47210,normal,20,1,5.000000e-01,-0.866025,0.866025,5.000000e-01,-0.500000,-0.40205,0.863100
2016-01-01 21:30:00+00:00,14.511001,92.811740,1017.37030,normal,21,1,7.071068e-01,-0.707107,0.866025,5.000000e-01,-0.449999,-0.10180,0.576870
2016-01-01 22:30:00+00:00,14.461000,92.507000,1016.67084,normal,22,1,8.660254e-01,-0.500000,0.866025,5.000000e-01,-0.050001,-0.69946,-0.304740
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31 13:30:00+00:00,17.900000,83.689150,1014.38940,normal,13,12,-9.659258e-01,-0.258819,1.000000,-2.449294e-16,-0.750000,0.59576,3.073070
2023-12-31 14:30:00+00:00,17.850000,82.082560,1014.78890,normal,14,12,-8.660254e-01,-0.500000,1.000000,-2.449294e-16,-0.050000,0.39950,-1.606590
2023-12-31 15:30:00+00:00,17.550000,81.780350,1014.98740,normal,15,12,-7.071068e-01,-0.707107,1.000000,-2.449294e-16,-0.300000,0.19850,-0.302210


In [82]:
df = df.drop(columns=['hour', 'month'])

In [83]:
df.isna().sum()

temperature          477
humidity             512
pressure             398
label                  0
cos_hour               0
sin_hour               0
cos_month              0
sin_month              0
temp_gradient        642
pressure_gradient    524
humidity_gradient    687
dtype: int64

In [85]:
df = df.drop(index="2016-01-01 18:30:00+00:00")
df

,temperature,humidity,pressure,label,cos_hour,sin_hour,cos_month,sin_month,temp_gradient,pressure_gradient,humidity_gradient
date,,,,,,,,,,,
2016-01-01 19:30:00+00:00,15.461000,91.371770,1017.87415,normal,0.258819,-0.965926,0.866025,5.000000e-01,-0.650000,-0.60199,1.130735
2016-01-01 20:30:00+00:00,14.961000,92.234870,1017.47210,normal,0.500000,-0.866025,0.866025,5.000000e-01,-0.500000,-0.40205,0.863100
2016-01-01 21:30:00+00:00,14.511001,92.811740,1017.37030,normal,0.707107,-0.707107,0.866025,5.000000e-01,-0.449999,-0.10180,0.576870
2016-01-01 22:30:00+00:00,14.461000,92.507000,1016.67084,normal,0.866025,-0.500000,0.866025,5.000000e-01,-0.050001,-0.69946,-0.304740
2016-01-01 23:30:00+00:00,15.111000,90.757430,1017.37310,normal,0.965926,-0.258819,0.866025,5.000000e-01,0.650000,0.70226,-1.749570
...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31 13:30:00+00:00,17.900000,83.689150,1014.38940,normal,-0.965926,-0.258819,1.000000,-2.449294e-16,-0.750000,0.59576,3.073070
2023-12-31 14:30:00+00:00,17.850000,82.082560,1014.78890,normal,-0.866025,-0.500000,1.000000,-2.449294e-16,-0.050000,0.39950,-1.606590
2023-12-31 15:30:00+00:00,17.550000,81.780350,1014.98740,normal,-0.707107,-0.707107,1.000000,-2.449294e-16,-0.300000,0.19850,-0.302210


In [86]:
df.to_csv("data/anomaly_labelled_data.csv")